In [1]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [2]:
# CONFIGURATION

ORDERS_PER_YEAR = 220_000
SHIPPED_PER_YEAR = 170_000
START_ORDER_NUM = 2000000

np.random.seed(42)
random.seed(42)

STORES = [
    ("Store A","55401",44.9833,-93.2667),  # MINNEAPOLIS, MN
    ("Store B","60601",41.8837,-87.6233),  # CHICAGO, IL
    ("Store C","48226",42.3314,-83.0458),  # DETROIT, MI
    ("Store D","15222",40.4406,-79.9959),  # PITTSBURGH, PA
    ("Store E","30303",33.7490,-84.3880),  # ATLANTA, GA
    ("Store F","37219",36.1627,-86.7816),  # NASHVILLE, TN
    ("Store G","32202",30.3322,-81.6557),  # JACKSONVILLE, FL
    ("Store H","33602",27.9506,-82.4572),  # TAMPA, FL
    ("Store I","28202",35.2271,-80.8431),  # CHARLOTTE, NC
    ("Store J","02108",42.3601,-71.0589),  # BOSTON, MA
]

DEPARTMENTS = [f"Dept_{i+1}" for i in range(25)]

In [3]:
# CATALOG GENERATION

def create_catalog(n_skus=43106):

    skus = [f"SKU{i:06}" for i in range(1, n_skus+1)]

    catalog = pd.DataFrame({
        "sku": skus,
        "sku_description": [f"sku_description_{i}" for i in range(1, n_skus+1)],
        "price": np.round(np.random.uniform(3,800,n_skus),2),
        "weight_lb": np.round(np.random.uniform(0.1,200,n_skus),2),
        "length_in": np.round(np.random.uniform(2,90,n_skus),1),
        "width_in": np.round(np.random.uniform(2,60,n_skus),1),
        "height_in": np.round(np.random.uniform(1,60,n_skus),1),
        "department": np.random.choice(DEPARTMENTS,n_skus)
    })

    # POPULARITY WEIGHT (LONG-TAIL DEMAND)
    catalog["popularity"] = np.random.pareto(1.5, n_skus)
    catalog["popularity"] = catalog["popularity"] / catalog["popularity"].sum()

    catalog.to_csv("product_catalog_1.csv", index=False)
    return catalog

In [4]:
# UTILITY FUNCTIONS

def random_date(year):
    start = datetime(year,1,1)
    end = datetime(year,12,31)
    return start + timedelta(days=random.randint(0,(end-start).days))


def random_zip():
    # GENERATES A RANDOM 5-DIGIT STRING ZIP WITHIN MAINLAND USA (EXCLUDING CA/AK/Islands)
    while True:
        # RANGE COVERS MA (01001) TO WA (99403)
        z_int = np.random.randint(1001, 99404)
        
        # EXCLUDE CA (90000-96199) AND AK (99500-99999)
        if not (90000 <= z_int <= 96199) and not (99500 <= z_int <= 99999):
            return str(z_int).zfill(5)

def random_latlon():
    return (
        np.round(np.random.uniform(25,49),6),
        np.round(np.random.uniform(-124,-67),6)
    )

In [5]:
# DIMENSION LOGIC

def dim_flag(weight, length, width, height, service):

    volume = length * width * height
    longest = max(length,width,height)

    if volume > 700 or longest > 74:
        return "Y"

    if service == "FedEx Home Delivery" and weight > 40:
        return "Y"

    if service == "FedEx Ground Economy" and weight > 149.99:
        return "Y"

    return "N"

In [6]:
# ORDER GENERATION

def generate_year(year, catalog, start_order):
    rows = []
    order_num = start_order
    num_lines_per_order = np.random.randint(1, 8, ORDERS_PER_YEAR)
    fulfillment_choices = np.random.choice(["BOPIS", "STH"], ORDERS_PER_YEAR, p=[0.35, 0.65])
    store_indices = np.random.randint(0, len(STORES), ORDERS_PER_YEAR)

    for i in range(ORDERS_PER_YEAR):
        n = num_lines_per_order[i]
        fulfillment = fulfillment_choices[i]
        store = STORES[store_indices[i]]
        order_date = random_date(year)
        items = catalog.sample(n, weights="popularity")
        
        if fulfillment == "BOPIS":
            ship_zip, lat, lon = store[1], store[2], store[3]
        else:
            ship_zip = random_zip()
            lat, lon = random_latlon()

        for line_idx, (_, item) in enumerate(items.iterrows(), start=1):
            qty = np.random.randint(1, 5)
            rows.append({
                "Order Number": order_num,
                "SKU": item.sku,
                "SKU Description": item.sku_description,
                "Prime Line Number": line_idx,
                "orderline_units_sold": qty,
                "Total without tax": round(item.price * qty, 2),
                "Fulfillment Method": fulfillment,
                "Ship Node Description": store[0],
                "Order Date": order_date,
                "Ship ZIP": ship_zip,
                "Ship Latitude": lat,
                "Ship Longitude": lon
            })
        order_num += 1

    df = pd.DataFrame(rows)
    shipped_orders = df["Order Number"].drop_duplicates().sample(SHIPPED_PER_YEAR)
    df["Order line status"] = np.where(df["Order Number"].isin(shipped_orders), "Shipped", "Cancelled")

    # DATE LOGIC
    ship_offsets = np.random.randint(1, 6, size=len(df))
    df["Shipment status Ship Date"] = pd.to_datetime(df["Order Date"]) + pd.to_timedelta(ship_offsets, unit='d')
    df.loc[df["Order line status"] == "Cancelled", "Shipment status Ship Date"] = pd.NaT
    
    # FORMAT DATES TO MM/DD/YYYY STRINGS
    df["Order Date"] = df["Order Date"].dt.strftime('%m/%d/%Y')
    df["Shipment status Ship Date"] = df["Shipment status Ship Date"].dt.strftime('%m/%d/%Y')

    df.to_csv(f"demand_{year}_1.csv", index=False)
    fulfilled = df[df["Order line status"] == "Shipped"].copy()
    fulfilled.to_csv(f"fulfilled_{year}_1.csv", index=False)

    return df, fulfilled, order_num

In [7]:
# SHIPPING & FEDEX LOGIC

def create_shipping(year, fulfilled):
    sth = fulfilled[fulfilled["Fulfillment Method"] == "STH"].copy()
    sth["Tracking Number"] = [f"TRK{year}{i:010}" for i in range(len(sth))]

    services = np.random.choice(["FedEx Home Delivery", "FedEx Ground Economy"], len(sth), p=[0.6, 0.4])
    sth["Service Description"] = services

    # DELIVERY DATE LOGIC
    delivery_offsets = np.where(
        sth["Service Description"] == "FedEx Home Delivery",
        np.random.randint(1, 6, size=len(sth)),
        np.random.randint(2, 10, size=len(sth))
    )
    
    # TEMPORARILY CONVERT BACK TO DATETIME TO DO THE MATH, THEN FORMAT TO STRING
    ship_dates_dt = pd.to_datetime(sth["Shipment status Ship Date"])
    sth["Delivery Date"] = (ship_dates_dt + pd.to_timedelta(delivery_offsets, unit='d')).dt.strftime('%m/%d/%Y')

    original_weight = np.round(np.random.uniform(1, 200, len(sth)), 2)
    length = np.round(np.random.uniform(5, 90, len(sth)), 1)
    width = np.round(np.random.uniform(5, 60, len(sth)), 1)
    height = np.round(np.random.uniform(2, 60, len(sth)), 1)

    dim_flags = [dim_flag(w, l, wid, h, s) for w,l,wid,h,s in zip(original_weight, length, width, height, services)]

    base_rate = np.random.uniform(0.6, 1.5, len(sth))
    service_multiplier = np.where(sth["Service Description"] == "FedEx Home Delivery", 1.15, 1.0)
    rated_weight = np.round(original_weight * np.random.uniform(1, 1.2, len(sth)), 2)
    net_charge = np.round((rated_weight * base_rate) * service_multiplier, 2)

    parcel = pd.DataFrame({
        "Shipment Tracking Number": sth["Tracking Number"],
        "Order Number": sth["Order Number"],
        "Recipient Postal Code": sth["Ship ZIP"],
        "Service Description": sth["Service Description"],
        "Ship Date": sth["Shipment status Ship Date"], # Matches the order file exactly
        "Delivery Date": sth["Delivery Date"],
        "Original Weight (Pounds)": original_weight,
        "Shipment Rated Weight (Pounds)": rated_weight,
        "Dimmed Height (in)": height,
        "Dimmed Width (in)": width,
        "Dimmed Length (in)": length,
        "Shipment DIM Flag": dim_flags,
        "Net Charge Amount USD": net_charge
    })

    parcel.to_csv(f"fedex_parcel_{year}_1.csv", index=False)
    sth[["Order Number","Tracking Number"]].to_csv(f"company_shipping_{year}_1.csv", index=False)

In [8]:
# STORES FILE

pd.DataFrame(STORES, columns=["Store","ZIP","Latitude","Longitude"]).to_csv("stores_1.csv", index=False)

In [9]:
# MAIN

catalog = create_catalog()

d2024, f2024, next_order = generate_year(2024, catalog, START_ORDER_NUM)
d2025, f2025, _ = generate_year(2025, catalog, next_order)

create_shipping(2024, f2024)
create_shipping(2025, f2025)

In [10]:
print("FULL ENTERPRISE DATASET GENERATED")

FULL ENTERPRISE DATASET GENERATED


In [11]:
# 10 OUTPUTS EXPECTED

# 2 REFERENCE: PRODUCT CATALOG, STORES
# 4 ORDERS: DEMAND 2024, DEMAND 2025, FULFILLED 2024, FULFILLED 2025
# 2 SHIPPING: COMPANY SHIPPING 2024, COMPANY SHIPPING 2025
# 2 PARCEL: FEDEX PARCEL 2024, FEDEX PARCEL 2025